<a href="https://colab.research.google.com/github/Asfia-Arpi/ML-Projects/blob/main/Breast_Cancer_Wisconsin_(Diagnostic).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS422 Lab Report – SVM Kernel Comparison
Student ID: **221-35-972**
Dataset: **Breast Cancer Wisconsin (Diagnostic)**

This notebook implements Linear, Polynomial, and RBF SVM models using GridSearchCV for hyperparameter tuning, evaluates them via cross-validation, and generates a PDF report.

In [ ]:
!pip install scikit-learn reportlab matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, RocCurveDisplay
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import cm
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.pdfgen import canvas

from reportlab.lib.utils import ImageReader
import os

## Step 1: Load Dataset

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Dataset shape: {X.shape}, Classes: {np.unique(y)}")

Dataset shape: (569, 30), Classes: [0 1]


## Step 2: Define Pipelines and Hyperparameter Grids

In [ ]:
pipelines = {
    "Linear": Pipeline([("scaler", StandardScaler()), ("svc", SVC(kernel="linear", probability=True, random_state=42))]),
    "Polynomial": Pipeline([("scaler", StandardScaler()), ("svc", SVC(kernel="poly", probability=True, random_state=42))]),
    "RBF": Pipeline([("scaler", StandardScaler()), ("svc", SVC(kernel="rbf", probability=True, random_state=42))]),
}

param_grids = {
    "Linear": {"svc__C": [0.01, 0.1, 1, 10, 100]},
    "Polynomial": {
        "svc__C": [0.1, 1, 10],
        "svc__degree": [2, 3],
        "svc__gamma": ["scale", "auto", 0.01, 0.1],
        "svc__coef0": [0.0, 0.5]
    },
    "RBF": {
        "svc__C": [0.1, 1, 10],
        "svc__gamma": ["scale", "auto", 0.01, 0.1]
    }
}

## Step 3: Hyperparameter Tuning with GridSearchCV

In [ ]:
best_estimators = {}
grid_results = {}
for name, pipe in pipelines.items():
    grid = GridSearchCV(pipe, param_grids[name], cv=cv, scoring="f1", n_jobs=-1)
    grid.fit(X, y)
    best_estimators[name] = grid.best_estimator_
    grid_results[name] = grid
    print(f"{name} Best Params: {grid.best_params_}")

Linear Best Params: {'svc__C': 0.1}
Polynomial Best Params: {'svc__C': 10, 'svc__coef0': 0.5, 'svc__degree': 3, 'svc__gamma': 0.01}
RBF Best Params: {'svc__C': 10, 'svc__gamma': 0.01}


## Step 4: Cross-validated Evaluation

In [ ]:
metrics_summary = []
conf_mats = {}
roc_fig_paths = {}
for name, model in best_estimators.items():
    y_pred = cross_val_predict(model, X, y, cv=cv, method="predict", n_jobs=-1)
    y_proba = cross_val_predict(model, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    rocauc = roc_auc_score(y, y_proba)
    cm = confusion_matrix(y, y_pred)
    metrics_summary.append({"Kernel": name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1, "ROC-AUC": rocauc})
    conf_mats[name] = cm
    fig, ax = plt.subplots()
    RocCurveDisplay.from_predictions(y, y_proba, ax=ax, name=f"{name} SVM")
    ax.set_title(f"ROC Curve - {name} SVM")
    fig_path = f"roc_{name.lower()}.png"
    plt.savefig(fig_path, bbox_inches="tight")
    plt.close()
    roc_fig_paths[name] = fig_path

results_df = pd.DataFrame(metrics_summary).sort_values(by="F1-Score", ascending=False)
results_df

,Kernel,Accuracy,Precision,Recall,F1-Score,ROC-AUC
1,Polynomial,0.982425,0.975342,0.997199,0.986150,0.996181
2,RBF,0.980668,0.977901,0.991597,0.984701,0.995521
0,Linear,0.975395,0.967302,0.994398,0.980663,0.994583


## Step 5: Optional - Bootstrap CI for Accuracy

In [ ]:
def bootstrap_ci_acc(y_true, y_pred, n_boot=500, random_state=42):
    rng = np.random.default_rng(random_state)
    accs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        accs.append(accuracy_score(y_true[idx], y_pred[idx]))
    return np.percentile(accs, 2.5), np.percentile(accs, 97.5)

ci_table = []
for name, model in best_estimators.items():
    y_pred = cross_val_predict(model, X, y, cv=cv, method="predict", n_jobs=-1)
    low, high = bootstrap_ci_acc(y, y_pred)
    ci_table.append({"Kernel": name, "Accuracy 95% CI (lower)": low, "Accuracy 95% CI (upper)": high})
ci_df = pd.DataFrame(ci_table)
ci_df

,Kernel,Accuracy 95% CI (lower),Accuracy 95% CI (upper)
0,Linear,0.963093,0.987698
1,Polynomial,0.971880,0.992970
2,RBF,0.970123,0.991213


from matplotlib import pyplot as plt
ci_df['Accuracy 95% CI (lower)'].plot(kind='hist', bins=20, title='Accuracy 95% CI (lower)')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
ci_df['Accuracy 95% CI (upper)'].plot(kind='hist', bins=20, title='Accuracy 95% CI (upper)')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
ci_df.groupby('Kernel').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
ci_df.plot(kind='scatter', x='Accuracy 95% CI (lower)', y='Accuracy 95% CI (upper)', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
ci_df['Accuracy 95% CI (upper)'].plot(kind='line', figsize=(8, 4), title='Accuracy 95% CI (upper)')
plt.gca().spines[['top', 'right']].set_visible(False)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(ci_df['Kernel'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(ci_df, x='Accuracy 95% CI (lower)', y='Kernel', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(ci_df['Kernel'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(ci_df, x='Accuracy 95% CI (upper)', y='Kernel', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)